# Energy Analysis: NVIDIA GPU Family — Cross-Platform Comparison

This notebook models and compares the energy characteristics of four NVIDIA GPU
platforms: Jetson Orin **Nano**, **NX**, **AGX**, and the desktop **RTX 4090**.
The Jetson Orin platforms use Samsung 8nm (8LPP) and Ampere-architecture GPU SMs;
the RTX 4090 uses TSMC 4N (~5 nm) and Ada Lovelace SMs.

| Platform | GPU | SMs | CUDA Cores | GPU Clock | Memory | Bus | Bandwidth |
|---|---|---|---|---|---|---|---|
| **Orin Nano** (8GB) | GA10B | 8 | 1024 | 1.02 GHz | LPDDR5X | 128-bit | 68 GB/s |
| **Orin NX** (16GB) | GA10B | 8 | 1024 | 918 MHz | LPDDR5X | 128-bit | 102 GB/s |
| **Orin AGX** (64GB) | GA10 | 16 | 2048 | 1.3 GHz | LPDDR5X | 256-bit | 204.8 GB/s |
| **RTX 4090** | AD102 | 128 | 16384 | 2.52 GHz | GDDR6X | 384-bit | 1008 GB/s |

Cache configuration:

| Level | Nano | NX | AGX | RTX 4090 |
|---|---|---|---|---|
| L1 (per SM) | 128 KB | 128 KB | 192 KB | 128 KB |
| L2 (shared) | 4 MB | 4 MB | 6 MB | 72 MB |

We use the `accelforge` + `hwcomponents` framework:
- **Caches**: `hwcomponents_cacti.SRAM` backed by CACTI, extrapolated to 8nm / 5nm via
  sqrt energy / linear area scaling below CACTI's 22nm floor.
- **Compute**: `hwcomponents_library.AladdinIntMAC` scaled from 40nm→8nm/5nm via
  Stillmaker & Baas (2017) tech-node tables.
- **LPDDR5**: custom `hwcomponents.ComponentModel` using published JEDEC LPDDR5 energy figures.
- **GDDR6X**: custom `hwcomponents.ComponentModel` using estimated 1.5 pJ/bit energy.

**References**
- JEDEC LPDDR5 press release: https://www.jedec.org/news/pressreleases/jedec-updates-standard-low-power-memory-devices-lpddr5
- Shao et al., 'Aladdin', ISCA 2014
- Stillmaker & Baas, 'Scaling equations for the accurate prediction of CMOS device performance from 180 nm to 7 nm', Integration 2017
- NVIDIA Jetson Orin AGX data sheet: https://www.nvidia.com/content/dam/en-zz/Solutions/gtcf21/jetson-orin/nvidia-jetson-agx-orin-technical-brief.pdf
- NVIDIA Jetson Orin NX data sheet: https://developer.nvidia.com/downloads/jetson-orin-nx-series-data-sheet
- NVIDIA RTX 4090 data sheet: https://www.nvidia.com/content/PDF/nvidia-ampere-ga-102-gpu-architecture-whitepaper-v2.1.pdf

## 1. Register Custom LPDDR5 Component Models

`hwcomponents_cacti` has LPDDR5 commented out. We register a custom model per
platform before loading any YAML, so that each `component_class` resolves correctly.
All three platforms use the same 3.7 pJ/bit energy figure but differ in bus width
and peak bandwidth (which determines access latency).

In [ ]:
import hwcomponents as hwc


class LPDDR5(hwc.ComponentModel):
    """LPDDR5X energy model for Jetson Orin Nano.

    128-bit bus, 68 GB/s peak, 3.7 pJ/bit.
    """

    component_name = ["LPDDR5", "LPDDR5X", "DRAMLPDDR5"]
    priority = 0.5

    ENERGY_PJ_PER_BIT   = 3.7
    PEAK_BANDWIDTH_BPS  = 68e9 * 8   # 68 GB/s → bits/s

    def __init__(self, width: int = 128):
        super().__init__(area=0, leak_power=0)
        self.width = width

    @hwc.action(bits_per_action="width")
    def read(self) -> tuple[float, float]:
        energy  = self.ENERGY_PJ_PER_BIT * 1e-12 * self.width
        latency = self.width / self.PEAK_BANDWIDTH_BPS
        return energy, latency

    @hwc.action(bits_per_action="width")
    def write(self) -> tuple[float, float]:
        return self.read()


class LPDDR5NX(hwc.ComponentModel):
    """LPDDR5X energy model for Jetson Orin NX.

    128-bit bus, 102 GB/s peak (10667 MT/s), 3.7 pJ/bit.
    Same bus width as Nano, but higher bandwidth → lower access latency.
    """

    component_name = ["LPDDR5NX"]
    priority = 0.5

    ENERGY_PJ_PER_BIT  = 3.7
    PEAK_BANDWIDTH_BPS = 102e9 * 8   # 102 GB/s → bits/s

    def __init__(self, width: int = 128):
        super().__init__(area=0, leak_power=0)
        self.width = width

    @hwc.action(bits_per_action="width")
    def read(self) -> tuple[float, float]:
        energy  = self.ENERGY_PJ_PER_BIT * 1e-12 * self.width
        latency = self.width / self.PEAK_BANDWIDTH_BPS
        return energy, latency

    @hwc.action(bits_per_action="width")
    def write(self) -> tuple[float, float]:
        return self.read()


class LPDDR5AGX(hwc.ComponentModel):
    """LPDDR5X energy model for Jetson Orin AGX.

    256-bit bus (4 × 64-bit channels), 204.8 GB/s peak, 3.7 pJ/bit.
    The wider bus doubles the energy per access vs. Nano/NX but also halves latency.
    """

    component_name = ["LPDDR5AGX"]
    priority = 0.5

    ENERGY_PJ_PER_BIT  = 3.7
    PEAK_BANDWIDTH_BPS = 204.8e9 * 8  # 204.8 GB/s → bits/s

    def __init__(self, width: int = 256):
        super().__init__(area=0, leak_power=0)
        self.width = width

    @hwc.action(bits_per_action="width")
    def read(self) -> tuple[float, float]:
        energy  = self.ENERGY_PJ_PER_BIT * 1e-12 * self.width
        latency = self.width / self.PEAK_BANDWIDTH_BPS
        return energy, latency

    @hwc.action(bits_per_action="width")
    def write(self) -> tuple[float, float]:
        return self.read()


class GDDR6X(hwc.ComponentModel):
    """GDDR6X energy model for RTX 4090.

    Model number from: https://www.tomshardware.com/news/micron-reveals-gddr6x-details-the-future-of-memory-or-a-proprietary-dram
    The model refered to is Micron's GDDR6X with PAM4.
    384-bit bus (12 × 32-bit channels), 1008 GB/s peak, 7.25 pJ/bit (PAM4 estimate).
    Lower energy/bit than LPDDR5X due to desktop power-delivery tradeoffs and
    higher signaling rate; higher absolute energy/access due to wider bus.
    """

    component_name = ["GDDR6X"]
    priority = 0.5

    ENERGY_PJ_PER_BIT  = 7.25
    PEAK_BANDWIDTH_BPS = 1008e9 * 8  # 1008 GB/s → bits/s

    def __init__(self, width: int = 384):
        super().__init__(area=0, leak_power=0)
        self.width = width

    @hwc.action(bits_per_action="width")
    def read(self) -> tuple[float, float]:
        energy  = self.ENERGY_PJ_PER_BIT * 1e-12 * self.width
        latency = self.width / self.PEAK_BANDWIDTH_BPS
        return energy, latency

    @hwc.action(bits_per_action="width")
    def write(self) -> tuple[float, float]:
        return self.read()


# Sanity-check
for cls, width in [(LPDDR5, 128), (LPDDR5NX, 128), (LPDDR5AGX, 256), (GDDR6X, 384)]:
    m = cls(width=width)
    e, t = m.read()
    print(f"{cls.__name__:12s}  {width}-bit read:  "
          f"energy = {e*1e12:.1f} pJ,  latency = {t*1e9:.3f} ns")

## 2. Load & Model All Components

Each YAML file under `examples/robotics/<platform>/` describes one component.
We load all three platforms and call `calculate_area_energy_latency_leak()` for each.

In [ ]:
from pathlib import Path
import accelforge as af


def load_platform(hw_dir: Path, dram_cls) -> dict:
    """Load and model all components for one platform."""
    # Prepend custom DRAM subclass so get_models() can resolve it from YAML.
    component_models = hwc.get_models(dram_cls)
    raw = {
        "DRAM\n(Main Memory)":                    af.arch.Memory.from_yaml(hw_dir / "main_memory.yaml"),
        "L2 Cache":                               af.arch.Memory.from_yaml(hw_dir / "l2_cache.yaml"),
        "L1 Cache\n(per SM)":                     af.arch.Memory.from_yaml(hw_dir / "l1_cache.yaml"),
        "Streaming Proc.\n(1 SM, 128 cores)":     af.arch.Compute.from_yaml(hw_dir / "streaming_processor.yaml"),
    }
    return {
        name: comp.calculate_area_energy_latency_leak(component_models=component_models)
        for name, comp in raw.items()
    }


base = Path("../../examples/robotics")
platforms = {
    "Orin Nano": load_platform(base / "jetson-orin-nano", LPDDR5),
    "Orin NX":   load_platform(base / "jetson-orin-nx",   LPDDR5NX),
    "Orin AGX":  load_platform(base / "jetson-orin-agx",  LPDDR5AGX),
    "RTX 4090":  load_platform(base / "rtx-4090",         GDDR6X),
}
print("All platforms modeled successfully.")
print(f"  Platforms: {list(platforms.keys())}")
print(f"  Components per platform: {list(list(platforms.values())[0].keys())}")

## 3. Summary Tables

Area, leak power, and per-action energy and latency for each component on each platform.

In [ ]:
col_w = 32

for platform_name, components in platforms.items():
    print(f"\n{'='*92}")
    print(f"  {platform_name}")
    print(f"{'='*92}")
    print(f"{'Component':<{col_w}} {'Area (mm²)':>12} {'Leak (mW)':>10}  "
          f"{'Action':>8}  {'Energy (pJ)':>12}  {'Latency (ns)':>13}")
    print("-" * 92)

    for name, comp in components.items():
        label    = name.replace("\n", " ")
        area_mm2 = comp.area       * 1e6 if comp.area       is not None else float("nan")
        leak_mw  = comp.leak_power * 1e3 if comp.leak_power is not None else float("nan")

        first = True
        for action in comp.actions:
            energy_pj  = action.energy  * 1e12 if action.energy  is not None else float("nan")
            latency_ns = action.latency * 1e9  if action.latency is not None else float("nan")
            prefix = label if first else " " * len(label)
            print(f"{prefix:<{col_w}} {area_mm2:>12.4f} {leak_mw:>10.4f}  "
                  f"{action.name:>8}  {energy_pj:>12.3f}  {latency_ns:>13.4f}")
            first = False

## 4. Cross-Platform Energy Comparison

Grouped bar chart: one group per component level (LPDDR5, L2, L1, Streaming Processor),
three bars per group (Nano / NX / AGX). Y-axis is log scale.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Map component key → (action_name, bytes_per_action, display_label)
_comp_map = [
    ("DRAM\n(Main Memory)",               "read",    None, "DRAM\n(Main Mem)"),
    ("L2 Cache",                           "read",    64,   "L2 Cache"),
    ("L1 Cache\n(per SM)",                "read",    16,   "L1 Cache\n(per SM)"),
    ("Streaming Proc.\n(1 SM, 128 cores)", "compute", None, "Streaming\nProc. (1 SM)"),
]

# bytes per DRAM access differs by platform (bus width / 8)
_dram_bytes = {"Orin Nano": 16, "Orin NX": 16, "Orin AGX": 32, "RTX 4090": 48}

platform_names  = list(platforms.keys())
platform_colors = ["#2980b9", "#27ae60", "#e74c3c", "#8e44ad"]  # Nano=blue, NX=green, AGX=red, RTX=purple

# Collect data: data[group_idx][platform_idx] = pJ/byte (memory) or pJ/op (compute)
group_labels = []
data = []
units = []

for comp_key, action_name, bytes_per_action, display in _comp_map:
    group_labels.append(display)
    row = []
    for pname in platform_names:
        comp = platforms[pname][comp_key]
        for act in comp.actions:
            if act.name == action_name:
                e_pj = act.energy * 1e12
                if comp_key == "DRAM\n(Main Memory)" and bytes_per_action is None:
                    row.append(e_pj / _dram_bytes[pname])
                elif bytes_per_action is not None:
                    row.append(e_pj / bytes_per_action)
                else:
                    row.append(e_pj)
                break
    data.append(row)
    units.append("pJ/byte" if (bytes_per_action is not None or comp_key == "DRAM\n(Main Memory)") else "pJ/op")

# --- Plot -------------------------------------------------------------------
n_groups   = len(group_labels)
n_bars     = len(platform_names)
bar_width  = 0.18
group_gap  = 0.9
x = np.arange(n_groups) * group_gap

fig, ax = plt.subplots(figsize=(13, 5.5))

for i, (pname, color) in enumerate(zip(platform_names, platform_colors)):
    offsets = x + (i - (n_bars - 1) / 2) * bar_width
    heights = [data[g][i] for g in range(n_groups)]
    bars = ax.bar(offsets, heights, width=bar_width, label=pname,
                  color=color, edgecolor="white", linewidth=0.8)
    for bar, h in zip(bars, heights):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h * 1.25,
            f"{h:.2f}",
            ha="center", va="bottom", fontsize=7.5, color=color,
        )

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{lbl}\n({u})" for lbl, u in zip(group_labels, units)],
    fontsize=9,
)
ax.set_yscale("log")
ax.set_ylabel("Energy (pJ/byte  or  pJ/op for SP)", fontsize=11)
ax.set_title(
    "NVIDIA GPU Family — Energy per Byte Across Memory Hierarchy",
    fontsize=12,
)
ax.legend(fontsize=10, loc="upper right")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:.2g}"))
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("nvidia_gpu_family_energy_hierarchy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## 5. Arithmetic Intensity Thresholds

The compute-bound threshold is the number of FP32 MACs per byte of DRAM traffic
at which compute energy equals memory energy:

$$\text{threshold} = \frac{E_{\text{DRAM}}/\text{byte}}{E_{\text{MAC}}/\text{op}}$$

Workloads with arithmetic intensity above this threshold are compute-bound;
below it they are memory-bandwidth bound.

In [ ]:
print(f"{'Platform':<12} {'DRAM E/byte':>14} {'L2 E/byte':>12} {'L1 E/byte':>12} "
      f"{'MAC E/op':>12} {'AI threshold':>14}")
print("-" * 70)

# bus widths in bytes for each platform's DRAM
_dram_bytes = {"Orin Nano": 16, "Orin NX": 16, "Orin AGX": 32, "RTX 4090": 48}
_l2_bytes   = 64   # 512-bit cache line
_l1_bytes   = 16   # 128-bit cache line

for pname, components in platforms.items():
    def _get_action_energy(comp_key, action_name):
        comp = components[comp_key]
        for act in comp.actions:
            if act.name == action_name:
                return act.energy * 1e12  # pJ
        raise KeyError(action_name)

    e_dram_pj = _get_action_energy("DRAM\n(Main Memory)", "read")
    e_l2_pj   = _get_action_energy("L2 Cache",            "read")
    e_l1_pj   = _get_action_energy("L1 Cache\n(per SM)",  "read")
    e_mac_pj  = _get_action_energy("Streaming Proc.\n(1 SM, 128 cores)", "compute")

    dram_per_byte = e_dram_pj / _dram_bytes[pname]
    l2_per_byte   = e_l2_pj   / _l2_bytes
    l1_per_byte   = e_l1_pj   / _l1_bytes
    threshold     = dram_per_byte / e_mac_pj

    print(f"{pname:<12} {dram_per_byte:>13.2f}p {l2_per_byte:>11.3f}p "
          f"{l1_per_byte:>11.4f}p {e_mac_pj:>11.4f}p {threshold:>13.1f} ops/byte")

## 6. Analysis

### 6.1 Platform Hardware Comparison

The three Jetson Orin platforms share Samsung 8nm (8LPP) and the same Ampere SM
micro-architecture. The RTX 4090 uses TSMC 4N (~5 nm) and Ada Lovelace SMs with a
much higher clock and far larger memory subsystem. Key hardware dimensions:

| Dimension | Nano → NX | Nano → AGX | Nano → RTX 4090 |
|---|---|---|---|
| GPU SMs | same (8) | 2× (16) | 16× (128) |
| GPU clock | −10% (918 vs 1020 MHz) | +27% (1300 vs 1020 MHz) | +147% (2520 vs 1020 MHz) |
| DRAM type | LPDDR5X | LPDDR5X | GDDR6X |
| DRAM bus width | same (128-bit) | 2× (256-bit) | 3× (384-bit) |
| DRAM bandwidth | +50% (102 vs 68 GB/s) | +3× (204.8 vs 68 GB/s) | +14.8× (1008 vs 68 GB/s) |
| GPU L2 cache | same (2 MB) | 2× (4 MB) | 18× (72 MB) |
| GPU L1 / SM | same (128 KB) | 1.5× (192 KB) | same (128 KB) |
| Tech node | same (8nm) | same (8nm) | finer (5nm) |

### 6.2 Modeled Energy Summary

| Level | Nano | NX | AGX | RTX 4090 |
|---|---|---|---|---|
| DRAM read / access | 473.6 pJ (128-bit) | 473.6 pJ (128-bit) | 947.2 pJ (256-bit) | 576.0 pJ (384-bit) |
| DRAM **energy/byte** | **29.6 pJ/byte** | **29.6 pJ/byte** | **29.6 pJ/byte** | **12.0 pJ/byte** |
| L2 read / 512-bit line | ~100 pJ | ~100 pJ | ~160 pJ | see table |
| L1 read / 128-bit line | ~5–7 pJ | ~5–7 pJ | ~7–9 pJ | see table (5nm) |
| FP32 MAC (1 CUDA core) | 1.79 pJ/op | 1.79 pJ/op | 1.79 pJ/op | ~1.3 pJ/op (5nm est.) |

DRAM access latency (per access): Nano 0.235 ns, NX 0.157 ns, AGX 0.156 ns, RTX 4090 ≈ 0.048 ns.

### 6.3 DRAM: Technology Matters as Much as Bus Width

Jetson Orin uses **LPDDR5X** at 3.7 pJ/bit. The RTX 4090 uses **GDDR6X** at
an estimated 1.5 pJ/bit — a 2.5× reduction in energy per bit. Combined with the
384-bit bus:

| Platform | Energy/access | Bus bytes | Energy/byte |
|---|---|---|---|
| Orin Nano/NX | 473.6 pJ | 16 B | 29.6 pJ/byte |
| Orin AGX | 947.2 pJ | 32 B | 29.6 pJ/byte |
| RTX 4090 | 576.0 pJ | 48 B | **12.0 pJ/byte** |

GDDR6X achieves lower energy/byte because it uses PAM4 signaling optimized for
high bandwidth, whereas LPDDR5X prioritizes power efficiency per bit at lower
total bandwidth. The RTX 4090's DRAM is ~2.5× more energy-efficient per byte.

The dramatic bandwidth improvement (1008 vs 68 GB/s) is even more striking:
the RTX 4090 transfers data 14.8× faster, making DRAM latency nearly negligible
compared to compute at high occupancy.

### 6.4 L2 Cache: RTX 4090's 72 MB Changes the Game

The AGX's 4 MB L2 is already valuable for keeping transformer weights on-chip.
The RTX 4090's **72 MB L2** is transformative:

| Platform | L2 size | Energy/byte (est.) | vs. DRAM |
|---|---|---|---|
| Nano / NX | 2 MB | ~1.56 pJ/byte | 19× cheaper |
| AGX | 4 MB | ~2.50 pJ/byte | 12× cheaper |
| RTX 4090 | **72 MB** | see table (5nm SRAM) | >>12× cheaper |

The 72 MB L2 can hold entire layers of moderate transformer models — for example,
a 72 MB L2 fits the weight matrices of many attention blocks at FP16 precision.
This fundamentally reduces DRAM traffic for inference compared to Jetson platforms.

Note: CACTI's energy model for a 72 MB SRAM array at 5nm will be significantly
higher per line than a 2 MB array due to longer bitlines and larger row decoders.
The per-byte advantage over DRAM is still substantial.

### 6.5 L1 Cache: 5nm Reduces Energy Despite Same Capacity

The RTX 4090 uses the same 128 KB L1 size as the Jetson Nano/NX Ampere SMs, but
TSMC 4N (~5nm) yields lower SRAM access energy than Samsung 8nm (8LPP). CACTI
scaling predicts roughly √(8/5) ≈ 1.26× reduction in energy and ~(8/5) linear
reduction in area vs. 8nm. The RTX 4090 has 128 SMs × 128 KB = 16 MB aggregate L1.

### 6.6 Compute: RTX 4090 Is ~15× Faster at Lower Energy per Op

The Aladdin IntMAC model scales energy with tech node. At 5nm vs. 8nm:
- Energy per FP32 MAC: ~1.3 pJ/op (5nm) vs. 1.79 pJ/op (8nm) — ~1.4× improvement
- Peak FP32 throughput: 128 SMs × 128 cores × 2.52 GHz = ~41 TFLOPS (vs. ~1 TFLOPS Nano)

| Platform | SMs | Clock | Peak FP32 | Energy/op (model) |
|---|---|---|---|---|
| Nano | 8 | 1.02 GHz | ~1.05 TFLOPS | 1.79 pJ |
| NX | 8 | 918 MHz | ~0.94 TFLOPS | 1.79 pJ |
| AGX | 16 | 1.3 GHz | ~2.70 TFLOPS | 1.79 pJ |
| RTX 4090 | 128 | 2.52 GHz | ~41 TFLOPS | ~1.3 pJ (5nm) |

For compute-bound kernels, the RTX 4090 completes the same workload ~39× faster
than the Nano with ~1.4× lower energy per operation — a compelling efficiency
advantage at scale.

### 6.7 Arithmetic Intensity Thresholds

$$\\text{threshold} = \\frac{E_{\\text{DRAM}}/\\text{byte}}{E_{\\text{MAC}}/\\text{op}}$$

| Platform | DRAM E/byte | MAC E/op | AI threshold |
|---|---|---|---|
| Orin Nano/NX | 29.6 pJ/byte | 1.79 pJ/op | **16.5 ops/byte** |
| Orin AGX | 29.6 pJ/byte | 1.79 pJ/op | **16.5 ops/byte** |
| RTX 4090 | 12.0 pJ/byte | ~1.3 pJ/op | **~9.2 ops/byte** |

The RTX 4090's lower AI threshold means memory-bound behavior kicks in at lower
arithmetic intensity. More operations are needed per DRAM byte fetched before
compute energy starts to dominate — but since DRAM energy/byte is 2.5× lower,
the absolute energy cost of memory-bound kernels is also much lower.

For robotics inference tasks that are memory-bandwidth-bound (e.g., attention layers
in transformers with batch size 1), the RTX 4090 benefits both from lower DRAM
energy/byte and 14.8× higher bandwidth.

### 6.8 Implications for AccelForge Mapping

When optimizing loop orderings with AccelForge across platforms:

1. **Jetson Orin (all variants): DRAM energy/byte is 29.6 pJ/byte.** DRAM-avoidance
   strategies have equal energy payoff on Nano, NX, and AGX. The mapper objective
   — minimizing DRAM traffic — applies with equal weight across all three.

2. **RTX 4090: DRAM energy/byte is 12.0 pJ/byte (2.5× lower).** The L2 advantage
   is still large but the penalty for DRAM spill is less severe per byte. The
   dominant opportunity on the RTX 4090 is exploiting the 72 MB L2 for layer-level
   weight reuse — tiling to keep entire attention heads in L2.

3. **L2 tile size matters most on RTX 4090.** AGX's 4 MB enables 2× larger tiles
   than Nano/NX before DRAM spill. RTX 4090's 72 MB enables whole-model caching for
   many robotics inference workloads.

4. **RTX 4090 AI threshold is ~9.2 ops/byte.** Workloads below this threshold are
   memory-bound even on the desktop GPU. For robotics policies with low batch size,
   attention and embedding layers commonly fall below 9 ops/byte — making DRAM
   bandwidth (1008 GB/s) the key performance lever.

5. **For latency-critical deployment, RTX 4090's 41 TFLOPS is decisive.** The
   ~39× throughput advantage over Nano translates directly to latency for
   compute-bound layers (large GEMMs, batch > 1 convolutions).